> Итог этапа 06: решения приняты. Актуальная выборка `06_05_cases_excluded.parquet` содержит 6 331 485 строк и 18 349 GoA=True. Расчеты ниже сохраняют исторический вход этого шага. Общий итог: `europe/reports/attempts/README.md`.


# 06_03 – Косвенные признаки для LSGG

Проверяем текущую выборку LSGG: обозначения ВПП по месяцам, последовательности попыток, интервалы, AP_C_RWY, повторяемость обозначений у тех же адресов и типов ВС. Все проверки ретроспективные и не формируют входы раннего прогноза.

Обозначения ВПП считаем один раз на наблюдение и обозначение; строка с двумя обозначениями входит в две группы. AP_C_RWY может происходить из того же обогащения и не является независимым подтверждением. Порог 39 секунд диагностический. Данные и метки не исправляем.

In [1]:
# Находим проект и проверяем косвенные признаки во всей выборке LSGG
from pathlib import Path
import sys
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'europe/scripts/audit_lsgg_indirect_evidence.py').exists())
sys.path.insert(0, str(ROOT / 'europe/scripts'))
import audit_lsgg_indirect_evidence as audit
case, results, journal = audit.main()
display(case)
display(results['interval_summary'])
display(results['runway_cross'])
print(journal)

,flight_id,icao24,callsign,typecode,airport,landing_time,attempt_times,ILS,AP_C_RWY,GoA,n_attempts,C40_CROSS_TIME,month
11233,TAP948_37820,495295,TAP948,A319,LSGG,2019-03-01 16:29:15+00:00,"[2019-03-01T16:29:03.000000, 2019-03-01T16:29:...","[22R, 22]",22,True,2,2019-03-01 16:06:06+00:00,2019-03


,pair_group,from_runway,to_runway,intervals,min_seconds,median_seconds,max_seconds,at_most_39_seconds,later_attempt_equals_landing
0,04L_04_pair,04L,04,1,732.0,732.0,732.0,0,0
1,22R_22_pair,22,22R,18,522.0,906.5,2094.0,0,2
2,22R_22_pair,22R,22,167,12.0,869.0,2123.0,1,63
3,other_change,04,22,4,634.0,944.5,1260.0,0,1
4,other_change,04,22R,2,727.0,751.0,775.0,0,0
5,other_change,22,04,4,364.0,791.0,896.0,0,1
6,other_change,22R,04,15,669.0,928.0,2014.0,0,3
7,same_label,04,04,171,397.0,948.0,1829.0,0,43
8,same_label,22,22,105,557.0,881.0,1777.0,0,41
9,same_label,22R,22R,45,407.0,827.0,1734.0,0,3


,runway_label,AP_C_RWY,observations,go_arounds
0,04,04,80474,189
1,04,22,293,6
2,04L,04,72,1
3,04L,22,1,0
4,22,04,189,8
5,22,22,54154,289
6,22R,04,317,19
7,22R,22,70084,228


{'input_sha256': '1918930dbbf8c410f36524a0f4c5c4e2d98c01ccafb1b2bcb2d579c1d6d70f72', 'input_unchanged': True, 'airport_rows': 205374, 'airport_go_arounds': 530, 'intervals': 532, 'pair_group_counts': {'same_label': 321, '22R_22_pair': 185, 'other_change': 25, '04L_04_pair': 1}, 'addresses_with_both_22_labels': 3606, 'addresses_checked': 6729, 'rows_changed': 0, 'rows_deleted': 0, 'historical_runway_status_verified': False, 'trajectory_validation_performed': False, 'AP_C_RWY_is_independent_evidence': False, 'threshold_39_seconds_is_diagnostic_only': True}


In [2]:
# Показываем месяцы и адреса с двумя обозначениями направления 22
display(results['monthly'].pivot(index='month',columns='runway_label',values='observations').fillna(0).astype(int))
display(results['address_overlap'][results['address_overlap'].has_both_22_labels].head(30))
# Полный перечень адресов и распределение типов сохранены в отчетах

runway_label,04,04L,22,22R
month,,,,
2019-01,3167,2,627,1783
2019-02,3038,5,652,1836
2019-03,2469,8,1068,3014
2019-04,2838,4,695,1791
2019-05,3797,4,640,1405
2019-06,2238,2,1203,2548
2019-07,2747,2,1315,2150
2019-08,2692,0,1161,1983
2019-09,2946,2,949,1714


,icao24,has_22,has_22R,has_both_22_labels
3,0100db,True,True,True
4,0100dc,True,True,True
5,0100dd,True,True,True
6,0100de,True,True,True
7,0100df,True,True,True
8,0100e0,True,True,True
12,0100ff,True,True,True
13,010108,True,True,True
14,010109,True,True,True
15,01010a,True,True,True
